In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Feature-Level Fusion – Liveness Detection
==========================================
Uses the spoof probability scores from each unimodal model as a feature vector and trains a meta-classifier on top.

This sits between score-level (weighted average) and deep fusion, and is the standard "stacking" / "late feature fusion" approach used in multimodal biometric papers.

Bimodal pairs evaluated:
  • Face + Voice
  • Face + Keystroke
  • Voice + Keystroke

Trimodal:
  • Face + Voice + Keystroke

Meta-classifiers used:
  • Logistic Regression  (simple, interpretable baseline)
  • Random Forest        (best for bimodal — handles non-linearity)
  • SVM (RBF kernel)     (strong with small feature spaces)
  • MLP                  (lightweight neural meta-learner)

Evaluation: 5-Fold Stratified Cross-Validation
  (chosen because n=95 is too small for a held-out test split)

Input  : fusion_val_predictions.csv

Outputs: feature_fusion_results.csv    (per-sample CV predictions)
         feature_fusion_summary.csv    (metrics per fusion combo × classifier)

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model        import LogisticRegression
from sklearn.svm                 import SVC
from sklearn.ensemble            import RandomForestClassifier
from sklearn.neural_network      import MLPClassifier
from sklearn.preprocessing       import StandardScaler
from sklearn.pipeline            import Pipeline
from sklearn.model_selection     import StratifiedKFold, cross_val_predict
from sklearn.metrics             import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

In [4]:
# ──────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ──────────────────────────────────────────────────────────────
df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/fusion_val_predictions.csv")
y  = df["true_label"].values

print("=" * 65)
print("FEATURE-LEVEL FUSION – LIVENESS DETECTION")
print("=" * 65)
print(f"Total samples    : {len(df)}")
print(f"Genuine (0)      : {(y == 0).sum()}")
print(f"Spoof   (1)      : {(y == 1).sum()}")
print(f"Attack types     : {df['attack_type'].value_counts().to_dict()}")
print(f"Evaluation       : 5-Fold Stratified Cross-Validation\n")

FEATURE-LEVEL FUSION – LIVENESS DETECTION
Total samples    : 95
Genuine (0)      : 24
Spoof   (1)      : 71
Attack types     : {'tts': 25, 'genuine': 24, 'logical': 17, 'replay': 15, 'synthetic': 14}
Evaluation       : 5-Fold Stratified Cross-Validation



In [5]:
# ──────────────────────────────────────────────────────────────
# 2.  FEATURE SETS  (one per fusion combination)
# ──────────────────────────────────────────────────────────────
# Each "feature vector" is the concatenation of per-modality
# spoof probabilities.  The meta-classifier learns a decision
# boundary in this probability space — more expressive than a
# fixed weighted average (score-level fusion).

FEATURE_SETS = {
    "Face + Voice": {
        "cols":   ["face_prob_spoof", "voice_prob_spoof"],
        "level":  "Bimodal",
    },
    "Face + Keystroke": {
        "cols":   ["face_prob_spoof", "keystroke_prob_spoof"],
        "level":  "Bimodal",
    },
    "Voice + Keystroke": {
        "cols":   ["voice_prob_spoof", "keystroke_prob_spoof"],
        "level":  "Bimodal",
    },
    "Trimodal (F+V+K)": {
        "cols":   ["face_prob_spoof", "voice_prob_spoof", "keystroke_prob_spoof"],
        "level":  "Trimodal",
    },
}

In [6]:
# ──────────────────────────────────────────────────────────────
# 3.  META-CLASSIFIERS
# ──────────────────────────────────────────────────────────────
CLASSIFIERS = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
    ),
    "SVM (RBF)": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(32, 16),
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
    ),
}


In [7]:
# ──────────────────────────────────────────────────────────────
# 4.  CROSS-VALIDATION EVALUATION
# ──────────────────────────────────────────────────────────────
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

summary_rows  = []
result_frames = []

for fusion_name, fset in FEATURE_SETS.items():

    X = df[fset["cols"]].values

    print(f"\n{'═' * 65}")
    print(f"  {fset['level'].upper()}  ▸  {fusion_name}")
    print(f"  Features : {fset['cols']}")
    print(f"{'═' * 65}")

    best_acc   = 0
    best_clf   = None
    best_preds = None
    best_probs = None

    for clf_name, clf in CLASSIFIERS.items():

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    clf),
        ])

        # Cross-validated predictions (all samples get an OOF prediction)
        y_pred = cross_val_predict(pipe, X, y, cv=SKF)
        y_prob = cross_val_predict(pipe, X, y, cv=SKF,
                                   method="predict_proba")[:, 1]

        acc  = accuracy_score(y, y_pred)
        prec = precision_score(y, y_pred,  zero_division=0)
        rec  = recall_score(y, y_pred,     zero_division=0)
        f1   = f1_score(y, y_pred,         zero_division=0)
        auc  = roc_auc_score(y, y_prob)
        cm   = confusion_matrix(y, y_pred)
        tn, fp, fn, tp = cm.ravel()
        far  = fp / (fp + tn) if (fp + tn) > 0 else 0
        frr  = fn / (fn + tp) if (fn + tp) > 0 else 0

        print(f"\n  ── {clf_name}")
        print(f"     Accuracy  : {acc*100:.2f}%")
        print(f"     Precision : {prec*100:.2f}%   Recall : {rec*100:.2f}%")
        print(f"     F1-Score  : {f1*100:.2f}%   AUC-ROC : {auc:.4f}")
        print(f"     FAR       : {far*100:.2f}%    FRR    : {frr*100:.2f}%")
        print(f"     Confusion : TN={tn} FP={fp} FN={fn} TP={tp}")

        summary_rows.append({
            "Level":          fset["level"],
            "Fusion Combo":   fusion_name,
            "Classifier":     clf_name,
            "Accuracy (%)":   round(acc  * 100, 2),
            "Precision (%)":  round(prec * 100, 2),
            "Recall (%)":     round(rec  * 100, 2),
            "F1-Score (%)":   round(f1   * 100, 2),
            "AUC-ROC":        round(auc, 4),
            "FAR (%)":        round(far  * 100, 2),
            "FRR (%)":        round(frr  * 100, 2),
            "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        })

        if acc > best_acc:
            best_acc   = acc
            best_clf   = clf_name
            best_preds = y_pred
            best_probs = y_prob

    # Best classifier full report
    print(f"\n  ★ Best classifier: {best_clf}  ({best_acc*100:.2f}%)")
    print(f"\n  Classification Report ({best_clf}):")
    print(classification_report(y, best_preds,
                                target_names=["Genuine", "Spoof"], digits=4))

    # Store best predictions for output CSV
    tmp = df[["session_token", "attack_type", "true_label"]].copy()
    tmp[f"feat_pred"]  = best_preds
    tmp[f"feat_prob"]  = best_probs.round(6)
    tmp["fusion"]      = fusion_name
    tmp["best_clf"]    = best_clf
    result_frames.append(tmp)


═════════════════════════════════════════════════════════════════
  BIMODAL  ▸  Face + Voice
  Features : ['face_prob_spoof', 'voice_prob_spoof']
═════════════════════════════════════════════════════════════════

  ── Logistic Regression
     Accuracy  : 92.63%
     Precision : 97.06%   Recall : 92.96%
     F1-Score  : 94.96%   AUC-ROC : 0.9812
     FAR       : 8.33%    FRR    : 7.04%
     Confusion : TN=22 FP=2 FN=5 TP=66

  ── Random Forest
     Accuracy  : 94.74%
     Precision : 95.83%   Recall : 97.18%
     F1-Score  : 96.50%   AUC-ROC : 0.9844
     FAR       : 12.50%    FRR    : 2.82%
     Confusion : TN=21 FP=3 FN=2 TP=69

  ── SVM (RBF)
     Accuracy  : 93.68%
     Precision : 97.10%   Recall : 94.37%
     F1-Score  : 95.71%   AUC-ROC : 0.9718
     FAR       : 8.33%    FRR    : 5.63%
     Confusion : TN=22 FP=2 FN=4 TP=67

  ── MLP
     Accuracy  : 76.84%
     Precision : 100.00%   Recall : 69.01%
     F1-Score  : 81.67%   AUC-ROC : 0.9636
     FAR       : 0.00%    FRR    : 30

In [8]:
# ──────────────────────────────────────────────────────────────
# 5.  COMPARISON vs SCORE-LEVEL & DECISION-LEVEL
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("FUSION METHOD COMPARISON  (all methods on same 95 samples)")
print("=" * 65)

# Score-level results (from fusion_val_predictions.csv)
score_level = {
    "Score: Face+Voice":          ("fv_pred",  "fv_prob_spoof"),
    "Score: Face+Keystroke":      ("fk_pred",  "fk_prob_spoof"),
    "Score: Voice+Keystroke":     ("vk_pred",  "vk_prob_spoof"),
    "Score: Trimodal (Eq.Wt.)":   ("fvk_pred", "fvk_prob_spoof"),
}

# Best feature-level result per combo (from summary)
feat_bests = {}
summary_df = pd.DataFrame(summary_rows)
for combo in FEATURE_SETS:
    subset = summary_df[summary_df["Fusion Combo"] == combo]
    best   = subset.loc[subset["Accuracy (%)"].idxmax()]
    feat_bests[f"Feature: {combo}"] = best

print(f"\n{'Method':<38} {'Acc':>7}  {'F1':>7}  {'AUC':>7}  {'FAR':>7}  {'FRR':>7}")
print("─" * 75)

# Score-level rows
for label, (pred_col, prob_col) in score_level.items():
    acc  = accuracy_score(y, df[pred_col])
    f1   = f1_score(y, df[pred_col], zero_division=0)
    auc  = roc_auc_score(y, df[prob_col])
    cm   = confusion_matrix(y, df[pred_col])
    tn, fp, fn, tp = cm.ravel()
    far  = fp / (fp + tn) if (fp + tn) > 0 else 0
    frr  = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f"{label:<38} {acc*100:>6.2f}%  {f1*100:>6.2f}%  {auc:>7.4f}"
          f"  {far*100:>6.2f}%  {frr*100:>6.2f}%")

print()

# Feature-level rows
for label, row in feat_bests.items():
    print(f"{label:<38} {row['Accuracy (%)']:>6.2f}%  {row['F1-Score (%)']:>6.2f}%"
          f"  {row['AUC-ROC']:>7.4f}  {row['FAR (%)']:>6.2f}%  {row['FRR (%)']:>6.2f}%"
          f"  ← {row['Classifier']}")



FUSION METHOD COMPARISON  (all methods on same 95 samples)

Method                                     Acc       F1      AUC      FAR      FRR
───────────────────────────────────────────────────────────────────────────
Score: Face+Voice                       92.63%   94.96%   0.9765    8.33%    7.04%
Score: Face+Keystroke                   95.79%   97.14%   0.9894    4.17%    4.23%
Score: Voice+Keystroke                  90.53%   93.53%   0.9783   12.50%    8.45%
Score: Trimodal (Eq.Wt.)                96.84%   97.87%   0.9959    4.17%    2.82%

Feature: Face + Voice                   94.74%   96.50%   0.9844   12.50%    2.82%  ← Random Forest
Feature: Face + Keystroke               97.89%   98.61%   0.9971    8.33%    0.00%  ← Random Forest
Feature: Voice + Keystroke              93.68%   95.83%   0.9487   16.67%    2.82%  ← Random Forest
Feature: Trimodal (F+V+K)               96.84%   97.87%   0.9947    4.17%    2.82%  ← Logistic Regression


In [9]:
import os

# ──────────────────────────────────────────────────────────────
# 6.  SAVE OUTPUTS
# ──────────────────────────────────────────────────────────────
output_dir = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions"
os.makedirs(output_dir, exist_ok=True)

results_df = pd.concat(result_frames, ignore_index=True)
results_df.to_csv(os.path.join(output_dir, "feature_fusion_results.csv"), index=False)

summary_df = summary_df.sort_values(
    ["Level", "Fusion Combo", "Accuracy (%)"],
    ascending=[True, True, False]
).reset_index(drop=True)
summary_df.to_csv(os.path.join(output_dir, "feature_fusion_summary.csv"), index=False)

print("\n")
print(f"✓  Saved: {output_dir}/feature_fusion_results.csv")
print(f"✓  Saved: {output_dir}/feature_fusion_summary.csv")
print("=" * 65)



✓  Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/feature_fusion_results.csv
✓  Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/feature_fusion_summary.csv
